# Train cGAN (WGAN-GP) with RFE2 — Colab GPU

**Setup:** IFS forecasts (tp, t2m, tcwv, sp) → RFE2 daily rainfall

**Before running:** Upload these folders to Google Drive:
- `CGAN/rfe_tfrecords/` — 16 training files + validation TFRecords
- `CGAN/dsrnngan/` — the code folder
- `CGAN/FCSTNorm2018.pkl`
- `CGAN/cGAN_data/` — elev.nc + lsm.nc
- `CGAN/IFS_training/` — raw IFS files (for validation plots)
- `CGAN/RFE/` — raw RFE2 files (for validation plots)

**Runtime:** Change to GPU via Runtime → Change runtime type → T4 GPU

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Paths matching the uploaded E:\CGAN folder structure on Drive
DRIVE_ROOT = '/content/drive/MyDrive/CGAN'

TFRECORDS_DIR = f'{DRIVE_ROOT}/rfe_tfrecords'
CODE_DIR = f'{DRIVE_ROOT}/SEWAA-forecasts/24h_accumulations/cGAN/dsrnngan'
CONSTANTS_DIR = f'{DRIVE_ROOT}/SEWAA-forecasts/cGAN_data'
FCSTNORM_DIR = f'{DRIVE_ROOT}/SEWAA-forecasts/24h_accumulations/cGAN'
IFS_DIR = f'{DRIVE_ROOT}/IFS_training'
RFE_DIR = f'{DRIVE_ROOT}/RFE'
LOG_FOLDER = f'{DRIVE_ROOT}/logs_RFE2_run03'

# Verify files exist
import glob
tf_files = glob.glob(f'{TFRECORDS_DIR}/*.tfrecords')
print(f'TFRecords: {len(tf_files)} files')
print(f'FCSTNorm: {os.path.exists(os.path.join(FCSTNORM_DIR, "FCSTNorm2018.pkl"))}')
print(f'Constants: {os.path.exists(CONSTANTS_DIR)}')
print(f'Code: {os.path.exists(CODE_DIR)}')

## 2. Install dependencies

In [ ]:
!pip install -q properscoring xarray netCDF4 pyyaml cartopy

## 3. Check GPU

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
if not gpus:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')

## 4. Set up paths and config

In [ ]:
import sys
sys.path.insert(0, CODE_DIR)

# Create data_paths.yaml for Colab
data_paths_content = f"""COLAB_RFE2:
    GENERAL:
        TRUTH_PATH: '{RFE_DIR}/'
        FORECAST_PATH: '{IFS_DIR}/'
        CONSTANTS_PATH: '{CONSTANTS_DIR}/'
        NORMALISATION_PATH: '{FCSTNORM_DIR}/'
        LEAD_IDX: 1
    TFRecords:
        tfrecords_path: '{TFRECORDS_DIR}/'
"""

with open(os.path.join(CODE_DIR, 'data_paths.yaml'), 'w') as f:
    f.write(data_paths_content)

# Create local_config.yaml for Colab
local_config_content = """data_paths: 'COLAB_RFE2'
gpu_mem_incr: True
use_gpu: True
disable_tf32: False
"""

with open(os.path.join(CODE_DIR, 'local_config.yaml'), 'w') as f:
    f.write(local_config_content)

print('Paths configured for Colab')
print(f'  Code: {CODE_DIR}')
print(f'  TFRecords: {TFRECORDS_DIR}')
print(f'  Constants: {CONSTANTS_DIR}')
print(f'  FCSTNorm: {FCSTNORM_DIR}')
print(f'  Logs: {LOG_FOLDER}')

## 5. Create training config

In [ ]:
import yaml

config = {
    'GENERAL': {
        'mode': 'GAN',
        'problem_type': 'normal',
    },
    'MODEL': {
        'architecture': 'forceconv',
        'padding': 'reflect',
    },
    'SETUP': {
        'log_folder': LOG_FOLDER,
    },
    'GENERATOR': {
        'filters_gen': 32,
        'noise_channels': 4,
        'latent_variables': 50,
        'learning_rate_gen': 1e-5,
    },
    'DISCRIMINATOR': {
        'filters_disc': 128,
        'learning_rate_disc': 1e-5,
    },
    'TRAIN': {
        'train_years': [2018, 2019, 2020, 2021],
        'training_weights': [0.4, 0.3, 0.2, 0.1],
        'num_samples': 204800,
        'steps_per_checkpoint': 128,
        'batch_size': 4,
        'kl_weight': 1e-8,
        'ensemble_size': 8,
        'CL_type': 'ensmeanMSE',
        'content_loss_weight': 1000.0,
    },
    'VAL': {
        'val_years': [2020],
        'val_size': 8,
    },
    'EVAL': {
        'num_batches': 25,
        'add_postprocessing_noise': True,
        'postprocessing_noise_factor': 1e-3,
        'max_pooling': True,
        'avg_pooling': True,
    },
}

config_path = os.path.join(CODE_DIR, 'config_colab.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config saved to {config_path}')
print(f'filters_gen: {config["GENERATOR"]["filters_gen"]}')
print(f'filters_disc: {config["DISCRIMINATOR"]["filters_disc"]}')
print(f'num_samples: {config["TRAIN"]["num_samples"]}')
print(f'batch_size: {config["TRAIN"]["batch_size"]}')
num_checkpoints = config['TRAIN']['num_samples'] // (config['TRAIN']['steps_per_checkpoint'] * config['TRAIN']['batch_size'])
print(f'Total checkpoints: {num_checkpoints}')

## 6. Import modules and load data

In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')

import data
import read_config
import setupdata
import setupmodel
import train

read_config.set_gpu_mode()
df_dict = read_config.read_downscaling_factor()

print(f'Fields: {data.all_fcst_fields}')
print(f'HOURS: {data.HOURS}')
print(f'Downscaling factor: {df_dict["downscaling_factor"]}')

In [ ]:
# Load config values
with open(config_path, 'r') as f:
    setup_params = yaml.safe_load(f)

mode = setup_params['GENERAL']['mode']
arch = setup_params['MODEL']['architecture']
padding = setup_params['MODEL']['padding']
log_folder = setup_params['SETUP']['log_folder']
filters_gen = setup_params['GENERATOR']['filters_gen']
lr_gen = float(setup_params['GENERATOR']['learning_rate_gen'])
noise_channels = setup_params['GENERATOR']['noise_channels']
latent_variables = setup_params['GENERATOR']['latent_variables']
filters_disc = setup_params['DISCRIMINATOR']['filters_disc']
lr_disc = float(setup_params['DISCRIMINATOR']['learning_rate_disc'])
train_years = setup_params['TRAIN']['train_years']
training_weights = setup_params['TRAIN']['training_weights']
num_samples = setup_params['TRAIN']['num_samples']
steps_per_checkpoint = setup_params['TRAIN']['steps_per_checkpoint']
batch_size = setup_params['TRAIN']['batch_size']
kl_weight = float(setup_params['TRAIN']['kl_weight'])
ensemble_size = setup_params['TRAIN']['ensemble_size']
CLtype = setup_params['TRAIN']['CL_type']
content_loss_weight = float(setup_params['TRAIN']['content_loss_weight'])
val_years = setup_params['VAL']['val_years']
val_size = setup_params['VAL']['val_size']
constant_fields = 2
input_channels = 2 * len(data.all_fcst_fields)  # 4 fields x 2 (mean + std) = 8

num_checkpoints = num_samples // (steps_per_checkpoint * batch_size)

print(f'Mode: {mode}, Arch: {arch}')
print(f'Input channels: {input_channels}')
print(f'Training: {num_samples} samples, {num_checkpoints} checkpoints')
print(f'Train years: {train_years}, Val years: {val_years}')

## 7. Set up data and model

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Verify FCSTNorm loaded
if data.fcst_norm is None:
    print('ERROR: FCSTNorm not loaded! Check NORMALISATION_PATH')
else:
    print(f'FCSTNorm loaded with {len(data.fcst_norm)} fields: {list(data.fcst_norm.keys())}')

# Verify data loading with validation dataset
sample = data_gen_valid.__getitem__(0)
lats = np.arange(-13.65, 24.65 + 0.1, 0.1)[:384]
lons = np.arange(19.15, 54.25 + 0.1, 0.1)[:352]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Elevation
ax = axes[0, 0]
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
mesh = ax.pcolormesh(lons, lats, sample[0]['hi_res_inputs'][0, :, :, 0], cmap='terrain_r', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.7)
ax.set_title('Elevation')

# Land-sea mask
ax = axes[0, 1]
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
mesh = ax.pcolormesh(lons, lats, sample[0]['hi_res_inputs'][0, :, :, 1], cmap='Blues', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.7)
ax.set_title('Land-sea mask')

# First forecast field (tp mean)
ax = axes[1, 0]
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
mesh = ax.pcolormesh(lons, lats, sample[0]['lo_res_inputs'][0, :, :, 0], cmap='Blues', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.7)
ax.set_title(f'IFS {data.all_fcst_fields[0]} (mean)')

# Truth (RFE2)
ax = axes[1, 1]
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
mesh = ax.pcolormesh(lons, lats, sample[1]['output'][0, :, :], cmap='Blues', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, shrink=0.7)
ax.set_title('RFE2 Truth')

plt.suptitle('Data Verification — check these look sensible before training', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nForecast shape: {sample[0]["lo_res_inputs"].shape}')
print(f'Constants shape: {sample[0]["hi_res_inputs"].shape}')
print(f'Truth shape: {sample[1]["output"].shape}')
print(f'Input channels: {sample[0]["lo_res_inputs"].shape[-1]} (expected {2 * len(data.all_fcst_fields)})')

## 6b. Verify FCSTNorm and data loading

Check that FCSTNorm loaded correctly and visualise elevation, LSM, forecast, and truth — same as Shruti's verification step.

In [ ]:
# Set up training and validation data
batch_gen_train, data_gen_valid = setupdata.setup_data(
    train_years=train_years,
    val_years=val_years,
    autocoarsen=False,
    weights=training_weights,
    batch_size=batch_size)

print('Data loaded')

In [ ]:
# Set up model
model = setupmodel.setup_model(
    mode=mode,
    arch=arch,
    downscaling_steps=df_dict['steps'],
    input_channels=input_channels,
    constant_fields=constant_fields,
    latent_variables=latent_variables,
    filters_gen=filters_gen,
    filters_disc=filters_disc,
    noise_channels=noise_channels,
    padding=padding,
    lr_disc=lr_disc,
    lr_gen=lr_gen,
    kl_weight=kl_weight,
    ensemble_size=ensemble_size,
    CLtype=CLtype,
    content_loss_weight=content_loss_weight)

print('Model ready')

## 8. Train!

Saves checkpoints to Google Drive so progress survives disconnections.

If Colab disconnects, re-run cells 1-7, then set `RESTART = True` below and re-run this cell to resume.

In [ ]:
# Set to True if resuming after a disconnection
RESTART = False

# Create log folder
Path(log_folder).mkdir(parents=True, exist_ok=True)
model_weights_root = os.path.join(log_folder, 'models')
Path(model_weights_root).mkdir(parents=True, exist_ok=True)

# Save config
with open(os.path.join(log_folder, 'setup_params.yaml'), 'w') as f:
    yaml.dump(setup_params, f, default_flow_style=False)

if RESTART:
    model.load(model.filenames_from_root(model_weights_root))
    with open(os.path.join(log_folder, 'run_status.json'), 'r') as f:
        run_status = json.load(f)
    training_samples = run_status['training_samples']
    checkpoint = int(training_samples / (steps_per_checkpoint * batch_size)) + 1
    log = pd.read_csv(os.path.join(log_folder, 'log.txt'))
    log_list = [log]
    print(f'Resumed from checkpoint {checkpoint}, {training_samples} samples')
else:
    training_samples = 0
    checkpoint = 1
    log_list = []

plot_fname = os.path.join(log_folder, 'progress.pdf')
log_file = os.path.join(log_folder, 'log.txt')
start_time = time.time()

while training_samples < num_samples:
    gc.collect()
    elapsed = (time.time() - start_time) / 60
    print(f'\nCheckpoint {checkpoint}/{num_checkpoints} | {training_samples}/{num_samples} samples | {elapsed:.0f} min elapsed')

    loss_log = train.train_model(
        model=model,
        mode=mode,
        batch_gen_train=batch_gen_train,
        data_gen_valid=data_gen_valid,
        noise_channels=noise_channels,
        latent_variables=latent_variables,
        checkpoint=checkpoint,
        steps_per_checkpoint=steps_per_checkpoint,
        num_cases=val_size,
        plot_fn=plot_fname)

    training_samples += steps_per_checkpoint * batch_size
    checkpoint += 1

    # Save to Drive (survives disconnections)
    model.save(model_weights_root)
    run_status = {'training_samples': training_samples}
    with open(os.path.join(log_folder, 'run_status.json'), 'w') as f:
        json.dump(run_status, f)

    log_data = {'training_samples': [training_samples]}
    for key in loss_log:
        log_data[key] = loss_log[key]
    log_list.append(pd.DataFrame(data=log_data))
    log = pd.concat(log_list)
    log.to_csv(log_file, index=False, float_format='%.6f')

    gen_weights_file = os.path.join(model_weights_root, f'gen_weights-{training_samples:07d}.h5')
    model.gen.save_weights(gen_weights_file)

total_time = (time.time() - start_time) / 3600
print(f'\nTraining complete! {total_time:.1f} hours')
print(f'Checkpoints saved to: {model_weights_root}')

## 9. Evaluate

In [ ]:
import evaluation

eval_fname = os.path.join(log_folder, 'eval_validation.txt')

interval = steps_per_checkpoint * batch_size
finalchkpt = num_samples // interval
# Evaluate last 4 checkpoints (blitz)
model_numbers = [(finalchkpt - ii) * interval for ii in range(3, -1, -1)] if finalchkpt >= 4 else [ii * interval for ii in range(1, finalchkpt + 1)]

print(f'Evaluating checkpoints: {model_numbers}')

evaluation.evaluate_multiple_checkpoints(
    mode=mode,
    arch=arch,
    val_years=val_years,
    log_fname=eval_fname,
    weights_dir=model_weights_root,
    autocoarsen=False,
    add_noise=True,
    noise_factor=1e-3,
    model_numbers=model_numbers,
    ranks_to_save=model_numbers,
    num_images=25,
    filters_gen=filters_gen,
    filters_disc=filters_disc,
    input_channels=input_channels,
    constant_fields=constant_fields,
    latent_variables=latent_variables,
    noise_channels=noise_channels,
    padding=padding,
    ensemble_size=10)

print('Evaluation complete')

## 10. Plot CRPS

In [ ]:
import matplotlib.pyplot as plt

checkpoints = []
crps = []

with open(eval_fname, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        try:
            checkpoints.append(int(parts[0]))
            crps.append(float(parts[1]))
        except (ValueError, IndexError):
            continue

plt.figure(figsize=(10, 6))
plt.plot(checkpoints, crps, 'o-', linewidth=2, markersize=6, color='steelblue')
plt.xlabel('Training Samples', fontsize=12)
plt.ylabel('CRPS', fontsize=12)
plt.title('CRPS vs Training Progress — RFE2 cGAN', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(log_folder, 'crps_plot.png'), dpi=150)
plt.show()
print(f'Final CRPS: {crps[-1]:.4f}')

## 11. View progress plots

In [ ]:
from IPython.display import display, Image as IPImage
import glob

# Show latest progress PDF as images
try:
    !apt-get install -y -q poppler-utils
    !pip install -q pdf2image
    from pdf2image import convert_from_path
    
    pdfs = sorted(glob.glob(os.path.join(log_folder, 'progress*.pdf')))
    if pdfs:
        images = convert_from_path(pdfs[-1])
        for img in images:
            display(img)
    else:
        print('No progress PDF found')
except Exception as e:
    print(f'Could not display PDF: {e}')
    print(f'Check {log_folder} on Drive manually')

## 12. View evaluation results

In [ ]:
if os.path.exists(eval_fname):
    with open(eval_fname, 'r') as f:
        print(f.read())
else:
    print('No evaluation results yet — run cell 9 first')